In [2]:
!nvidia-smi

Fri Mar 27 14:32:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   24C    P8              1W /  250W |    3490MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%pip install numpy
%pip install torch torchvision
%pip install wandb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


In [5]:
from torchvision.datasets import MNIST, SVHN

# Transforms for MNIST (grayscale 28x28 → resize to 32x32, convert to 3 channels)
mnist_transform = transforms.Compose([
    transforms.Resize((32, 32)),          # resize to 32x32 to match SVHN later
    transforms.Grayscale(num_output_channels=3),  # convert 1 channel → 3 channels
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and load MNIST
train_mnist = MNIST(root='./data', train=True, download=True, transform=mnist_transform)
test_mnist  = MNIST(root='./data', train=False, download=True, transform=mnist_transform)

train_loader_mnist = DataLoader(train_mnist, batch_size=64, shuffle=True)
test_loader_mnist  = DataLoader(test_mnist,  batch_size=64, shuffle=False)

print("MNIST train size:", len(train_mnist))
print("MNIST test size:", len(test_mnist))

100.0%
100.0%
100.0%
100.0%

MNIST train size: 60000
MNIST test size: 10000


In [6]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3 channels because we converted MNIST to RGB
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 32x32 → 16x16

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 16x16 → 8x8

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 8x8 → 4x4
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize model and move to GPU
model = CNN(num_classes=10).to(device)
print(model)
print("\nTotal parameters:", sum(p.numel() for p in model.parameters()))

CNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=2048, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.

In [10]:
import wandb
wandb.login(key="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", relogin=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: swagatambsws-biswas (swagatambsws-biswas-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [12]:
model = train_model(
    model=model,
    train_loader=train_loader_mnist,
    test_loader=test_loader_mnist,
    epochs=10,
    lr=0.001,
    run_name="mnist_training"
)

Epoch [1/10] Train Loss: 0.1502 | Train Acc: 95.47% | Test Loss: 0.0340 | Test Acc: 98.86%
Epoch [2/10] Train Loss: 0.0641 | Train Acc: 98.08% | Test Loss: 0.0249 | Test Acc: 99.16%
Epoch [3/10] Train Loss: 0.0494 | Train Acc: 98.53% | Test Loss: 0.0238 | Test Acc: 99.25%
Epoch [4/10] Train Loss: 0.0444 | Train Acc: 98.70% | Test Loss: 0.0230 | Test Acc: 99.28%
Epoch [5/10] Train Loss: 0.0382 | Train Acc: 98.89% | Test Loss: 0.0237 | Test Acc: 99.11%
Epoch [6/10] Train Loss: 0.0336 | Train Acc: 99.00% | Test Loss: 0.0270 | Test Acc: 99.16%
Epoch [7/10] Train Loss: 0.0259 | Train Acc: 99.19% | Test Loss: 0.0193 | Test Acc: 99.39%
Epoch [8/10] Train Loss: 0.0231 | Train Acc: 99.28% | Test Loss: 0.0205 | Test Acc: 99.41%
Epoch [9/10] Train Loss: 0.0238 | Train Acc: 99.28% | Test Loss: 0.0169 | Test Acc: 99.48%
Epoch [10/10] Train Loss: 0.0176 | Train Acc: 99.42% | Test Loss: 0.0255 | Test Acc: 99.36%

Final MNIST Test Accuracy: 99.36%


epoch,▁▂▃▃▄▅▆▆▇█
test_accuracy,▁▄▅▆▄▄▇▇█▇
test_loss,█▄▄▃▄▅▂▂▁▅
train_accuracy,▁▆▆▇▇▇████
train_loss,█▃▃▂▂▂▁▁▁▁
epoch,10
test_accuracy,99.36
test_loss,0.02551
train_accuracy,99.42333
train_loss,0.01759
